# **Build SQLite database from invoice CSV files**

This notebook reads three CSV files:
- `invoice_headers.csv`
- `invoice_line_items.csv`
- `invoice_summaries.csv`

It creates a SQLite database with three linked tables:
- `invoices`
- `invoice_items`
- `invoice_summary`


In [1]:
from pathlib import Path
import sqlite3
import pandas as pd


OUTPUT_DIR = Path("/home/devkumar-patel/Personal/sql_agent/database")
INPUT_DIR = Path("/home/devkumar-patel/Personal/sql_agent/data_csv")
DB_NAME = "invoices.db"

HEADERS_CSV = INPUT_DIR / "invoice_headers.csv"
ITEMS_CSV = INPUT_DIR / "invoice_line_items.csv"
SUMMARY_CSV = INPUT_DIR / "invoice_summaries.csv"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = OUTPUT_DIR / DB_NAME

print(f"Database will be created at: {DB_PATH}")

Database will be created at: /home/devkumar-patel/Personal/sql_agent/database/invoices.db


In [2]:
# Load the CSV files
headers_df = pd.read_csv(HEADERS_CSV)
items_df = pd.read_csv(ITEMS_CSV)
summary_df = pd.read_csv(SUMMARY_CSV)

print('Headers rows:', len(headers_df))
print('Items rows:', len(items_df))
print('Summary rows:', len(summary_df))

display(headers_df.head())
display(items_df.head())
display(summary_df.head())

Headers rows: 50
Items rows: 213
Summary rows: 50


,invoice_no,date_of_issue,seller_name,seller_address,seller_tax_id,seller_gstin,client_name,client_address,client_tax_id
0,51109301,03/07/2023,TechVision Distributors Pvt Ltd,"Plot 14, MIDC Industrial Area, Andheri East, M...",27AABCT1234F1Z5,27AABCT1234F1Z5,Raj Electronics Pvt Ltd,"42 MG Road, Bengaluru, Karnataka - 560001",901-95-4704
1,51109302,25/09/2023,TechVision Distributors Pvt Ltd,"Plot 14, MIDC Industrial Area, Andheri East, M...",27AABCT1234F1Z5,27AABCT1234F1Z5,Sharma Tech Solutions,"15 Connaught Place, New Delhi, Delhi - 110001",322-76-5821
2,51109303,12/08/2023,TechVision Distributors Pvt Ltd,"Plot 14, MIDC Industrial Area, Andheri East, M...",27AABCT1234F1Z5,27AABCT1234F1Z5,Mumbai Gadget House,"78 Linking Road, Mumbai, Maharashtra - 400050",809-21-3533
3,51109304,07/10/2023,TechVision Distributors Pvt Ltd,"Plot 14, MIDC Industrial Area, Andheri East, M...",27AABCT1234F1Z5,27AABCT1234F1Z5,Chennai Digital Store,"23 Anna Salai, Chennai, Tamil Nadu - 600002",235-81-3906
4,51109305,09/03/2024,TechVision Distributors Pvt Ltd,"Plot 14, MIDC Industrial Area, Andheri East, M...",27AABCT1234F1Z5,27AABCT1234F1Z5,Hyderabad IT Traders,"56 Jubilee Hills, Hyderabad, Telangana - 500033",570-19-4743


,invoice_no,item_no,description,qty,unit,net_price,net_worth,vat_pct,gross_worth
0,51109301,1,Garmin Fenix 7 Solar Multisport GPS,9.0,pcs,74120.0,667080.0,10%,733788.0
1,51109301,2,Apple Watch Series 9 GPS 45mm Midnight,8.0,pcs,52083.0,416664.0,10%,458330.4
2,51109301,3,Xiaomi 14 Pro 512GB White,8.0,pcs,74154.0,593232.0,10%,652555.2
3,51109302,1,Apple MacBook Air M2 8GB 256GB Silver,7.0,pcs,102933.0,720531.0,10%,792584.1
4,51109302,2,TP-Link Archer AX73 WiFi 6 Router,5.0,pcs,15075.0,75375.0,10%,82912.5


,invoice_no,vat_pct,total_net_worth,total_vat,total_gross_worth
0,51109301,10%,1676976.0,167697.6,1844673.6
1,51109302,10%,1780513.0,178051.3,1958564.3
2,51109303,10%,956835.0,95683.5,1052518.5
3,51109304,10%,1704063.0,170406.3,1874469.3
4,51109305,10%,2023625.0,202362.5,2225987.5


In [3]:
# Basic cleanup / normalization
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    return df

headers_df = normalize_columns(headers_df)
items_df = normalize_columns(items_df)
summary_df = normalize_columns(summary_df)

# Make sure invoice_no stays as text for consistent joins and keys
headers_df['invoice_no'] = headers_df['invoice_no'].astype(str)
items_df['invoice_no'] = items_df['invoice_no'].astype(str)
summary_df['invoice_no'] = summary_df['invoice_no'].astype(str)

# Optional: force other identifier-like fields to text
for col in ['seller_tax_id', 'seller_gstin', 'client_tax_id']:
    if col in headers_df.columns:
        headers_df[col] = headers_df[col].astype(str)

print('Normalized columns ready.')

Normalized columns ready.


In [4]:
# Create SQLite database and tables
conn = sqlite3.connect(DB_PATH)
conn.execute('PRAGMA foreign_keys = ON;')
cur = conn.cursor()

# Drop tables if they already exist so reruns are clean
cur.execute('DROP TABLE IF EXISTS invoice_items;')
cur.execute('DROP TABLE IF EXISTS invoice_summary;')
cur.execute('DROP TABLE IF EXISTS invoices;')

cur.execute('''
CREATE TABLE invoices (
    invoice_no TEXT PRIMARY KEY,
    date_of_issue TEXT,
    seller_name TEXT,
    seller_address TEXT,
    seller_tax_id TEXT,
    seller_gstin TEXT,
    client_name TEXT,
    client_address TEXT,
    client_tax_id TEXT
);
''')

cur.execute('''
CREATE TABLE invoice_items (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    invoice_no TEXT NOT NULL,
    item_no INTEGER,
    description TEXT,
    qty REAL,
    unit TEXT,
    net_price REAL,
    net_worth REAL,
    vat_pct TEXT,
    gross_worth REAL,
    FOREIGN KEY (invoice_no) REFERENCES invoices(invoice_no)
        ON UPDATE CASCADE
        ON DELETE CASCADE
);
''')

cur.execute('''
CREATE TABLE invoice_summary (
    invoice_no TEXT PRIMARY KEY,
    vat_pct TEXT,
    total_net_worth REAL,
    total_vat REAL,
    total_gross_worth REAL,
    FOREIGN KEY (invoice_no) REFERENCES invoices(invoice_no)
        ON UPDATE CASCADE
        ON DELETE CASCADE
);
''')

conn.commit()
print('Tables created successfully.')

Tables created successfully.


In [5]:
# Insert parent table first
invoice_cols = [
    'invoice_no', 'date_of_issue', 'seller_name', 'seller_address',
    'seller_tax_id', 'seller_gstin', 'client_name', 'client_address', 'client_tax_id'
]
missing_headers = [c for c in invoice_cols if c not in headers_df.columns]
if missing_headers:
    raise ValueError(f'Missing columns in invoice_headers.csv: {missing_headers}')

headers_df[invoice_cols].to_sql('invoices', conn, if_exists='append', index=False)
print(f'Inserted {len(headers_df)} invoice header rows.')

Inserted 50 invoice header rows.


In [6]:
# Insert child table: invoice_items
items_cols = [
    'invoice_no', 'item_no', 'description', 'qty', 'unit',
    'net_price', 'net_worth', 'vat_pct', 'gross_worth'
]
missing_items = [c for c in items_cols if c not in items_df.columns]
if missing_items:
    raise ValueError(f'Missing columns in invoice_line_items.csv: {missing_items}')

# Convert item_no to integer where possible
items_df['item_no'] = pd.to_numeric(items_df['item_no'], errors='coerce').astype('Int64')

items_df[items_cols].to_sql('invoice_items', conn, if_exists='append', index=False)
print(f'Inserted {len(items_df)} invoice item rows.')

Inserted 213 invoice item rows.


In [7]:
# Insert child table: invoice_summary
summary_cols = ['invoice_no', 'vat_pct', 'total_net_worth', 'total_vat', 'total_gross_worth']
missing_summary = [c for c in summary_cols if c not in summary_df.columns]
if missing_summary:
    raise ValueError(f'Missing columns in invoice_summaries.csv: {missing_summary}')

summary_df[summary_cols].to_sql('invoice_summary', conn, if_exists='append', index=False)
print(f'Inserted {len(summary_df)} invoice summary rows.')

Inserted 50 invoice summary rows.


In [8]:
# Verify table counts and sample joins
counts = {}
for table in ['invoices', 'invoice_items', 'invoice_summary']:
    counts[table] = pd.read_sql_query(f'SELECT COUNT(*) AS row_count FROM {table}', conn)['row_count'][0]

print(counts)

sample = pd.read_sql_query('''
SELECT i.invoice_no, i.date_of_issue, s.total_gross_worth, COUNT(it.id) AS item_count
FROM invoices i
LEFT JOIN invoice_summary s ON s.invoice_no = i.invoice_no
LEFT JOIN invoice_items it ON it.invoice_no = i.invoice_no
GROUP BY i.invoice_no, i.date_of_issue, s.total_gross_worth
ORDER BY i.invoice_no
LIMIT 10;
''', conn)
display(sample)

conn.close()
print('Done.')

{'invoices': np.int64(50), 'invoice_items': np.int64(213), 'invoice_summary': np.int64(50)}


,invoice_no,date_of_issue,total_gross_worth,item_count
0,51109301,03/07/2023,1844673.6,3
1,51109302,25/09/2023,1958564.3,6
2,51109303,12/08/2023,1052518.5,4
3,51109304,07/10/2023,1874469.3,6
4,51109305,09/03/2024,2225987.5,6
5,51109306,02/02/2024,264140.8,1
6,51109307,12/01/2024,742146.9,5
7,51109308,31/07/2023,971584.9,4
8,51109309,15/04/2023,307144.2,1
9,51109310,03/08/2023,1330274.0,4


Done.
